# DLFE 3-Flow 실습 랩 — 실데이터·실모델·사전학습 가중치

이 노트북은 **Ding et al. (IJCAI 2015) 이벤트 기반 주가 예측** 재현 프로젝트를,
실제로 이 저장소에 있는 **진짜 데이터(artifacts/)와 사용자가 학습시킨 진짜 가중치**로 따라가는 실습입니다.

| Flow | 내용 | 관련 스크립트 |
|---|---|---|
| **Flow 1** | 논문 완전 재현 — 뉴스→이벤트→임베딩→CNN→백테스트 | `flows/flow1_paper_reproduction/` (s1~s9) |
| **Flow 2** | 업그레이드 1 — FinBERT/피처/워크포워드로 US 신호 한계 진단 | `flows/flow2_us_upgrade/` (t1·t2, s10~s27, s37, s53) |
| **Flow 3** | 업그레이드 2 — KR + HIGH/LOW 타깃 재정의 + 실거래성 검증 | `flows/flow3_kr_highlow/` (s28~s36, s39~s55) |

**실행 원칙**
- 커널: `D:\Github\homeserver\alphamale\.venv\Scripts\python.exe` (pandas/torch/matplotlib 포함).
- 무거운 원본 학습(s1~s9 전체 체인)은 **다시 돌리지 않습니다**. 이미 생성된 `artifacts/`와
  **사전학습 가중치(`ntn.pt`)·저장된 추론 확률**로 추론/백테스트합니다.
- 짧은 학습 데모(9번 셀)만 실제로 몇 에폭 돌립니다 (CPU 수십 초).
- 시각화 코드는 전부 `src/dlfe_lab/viz.py`에 있고, 노트북은 **불러서 쓰기만** 합니다.

In [ ]:
# [설정] src/dlfe_lab 를 import 경로에 넣고, flow 스크립트(s4_ntn 등)를 쓸 수 있게 bootstrap 합니다.
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib
try:
    get_ipython()            # Jupyter면 inline 백엔드 유지
except NameError:
    matplotlib.use("Agg")    # 스크립트 실행이면 화면 없이 렌더

import numpy as np
import pandas as pd

from dlfe_lab import paths, data, embeddings, modeling, backtest, kr, viz
paths.bootstrap()

print("ROOT  =", paths.ROOT)
print("ART   =", paths.ART)
print("flows =", [p.name for p in paths.FLOW_DIRS])

---
## Flow 1 — 논문 완전 재현 (실데이터)

### 1-1. 내가 다루는 뉴스 데이터는 무엇인가?

아래 셀은 `artifacts/news.parquet`(s1_data.py 산출물)를 열어서
- 기사 수 / 티커 구성 / 기간
- 제목 토큰 길이 분포
- 실제 NVDA 헤드라인 5개

를 확인합니다. **논문의 "large-scale financial news corpus"가 이 프로젝트에서는
FMP 뉴스 제목 9개 메가캡 코퍼스**라는 사실을 눈으로 확인하는 단계입니다.

In [ ]:
news = data.load_news()
st = data.news_stats(news)
print(f"기사 수      : {st['rows']:,}")
print(f"티커 수      : {st['tickers']}  기간: {st['date_min']} ~ {st['date_max']}")
print(f"토큰 길이    : mean {st['tokens_mean']:.1f} / median {st['tokens_p50']:.0f} / max {st['tokens_max']}")
print(f"임베딩 학습용(pre-test) 비율: {st['emb_eligible_ratio']:.1%}")
print("\n티커별 기사 수:")
print(st["per_ticker"].to_string())
print("\nNVDA 실제 헤드라인 5개:")
for t in news[news.ticker == "NVDA"].title_clean.head(5):
    print("  -", t)
viz.news_overview(news)

### 1-2. 가격/라벨 데이터 — 무엇을 맞추는 문제인가?

`prices.parquet`에는 NVDA 일봉과 논문식 라벨(다음날 종가 방향 ±1),
그리고 시간순 train/dev/test split이 들어 있습니다.
**상승비율(=majority baseline)이 이미 0.53~0.54 근처**라는 점을 기억하세요 —
이 숫자가 Flow 2 내내 "정확도의 천장"으로 다시 등장합니다.

In [ ]:
prices = data.load_prices()
ps = data.price_stats(prices)
for split in ("train", "dev", "test"):
    s = ps[split]
    print(f"{split:5s}: {s['days']:4d}일  {s['start']} ~ {s['end']}  상승비율 {s['up_rate']:.3f}")
viz.price_overview(prices)

### 1-3. 이벤트 추출 결과 — 제목이 (O1, P, O2) 삼중항으로

s3_events.py는 spaCy 의존구문분석으로 제목에서 **주어(O1)-동사(P)-목적어(O2)**를 뽑습니다.
아래에서 NVDA 이벤트 원문 8개를 직접 읽어 보세요.
"몇 %의 이벤트가 임베딩 사전(vocab)에 완전히 커버되는가(in-vocab)"도 중요한 품질 지표입니다.

In [ ]:
ev = data.load_events()
ok = data.load_event_ok()
es = data.event_stats(ev, ok)
print(f"이벤트 수: {es['rows']:,}  | in-vocab {es['in_vocab']:,} ({es['in_vocab_ratio']:.1%})")
print("\n티커별 이벤트 수:")
print(es["per_ticker"].to_string())
print("\nNVDA SVO 이벤트 8개:")
nv_ok = np.where(ok & (ev.ticker.values == "NVDA"))[0]
for i in nv_ok[:8]:
    print(f"  {pd.Timestamp(ev.date.iloc[int(i)]).date()}  {data.triple_str(ev, int(i))}")

### 1-4. 단어 임베딩 (skip-gram, d=100) — 벡터를 직접 보기

s2_word2vec.py가 학습한 `word_vectors.npz`(7,024 단어 × 100차원)를 열어
**cosine 최근접 이웃**과 **PCA 2D 산점도**로 임베딩 공간을 확인합니다.
'nvidia' 근처에 무엇이 있는지, 방향성 동사들('soars', 'falls')이 어떻게 모이는지 보세요.

In [ ]:
vocab, w2i, W = embeddings.load_word_vectors()
print("vocab:", len(vocab), "| dim:", W.shape[1])
for q in ("nvidia", "soars", "falls"):
    try:
        nb = embeddings.nearest_words(q, k=6)
        print(f"\n'{q}' 이웃: " + ", ".join(f"{w}({c:+.2f})" for w, c in nb))
    except KeyError as e:
        print(f"\n{e}")
xy = embeddings.pca_2d(W[:400])
labels = [vocab[i] if i < 30 else None for i in range(400)]
viz.embedding_scatter(xy, labels=labels, title="skip-gram 단어 임베딩 PCA (상위 400 단어, 30개 라벨)")

### 1-5. NTN(Neural Tensor Network) — 모델이 어떻게 생겼는가 + 사전학습 가중치 로드

논문 Eq.1: `R1 = f(O1ᵀT1P + W1[O1;P] + b1)`, `R2 = f(PᵀT2O2 + …)`, `U = f(R1ᵀT3R2 + …)`.
아래 셀은 **실제 모델 클래스(s4_ntn.NTN)** 를 출력하고,
**사용자가 학습시켜 저장한 `artifacts/ntn.pt` 가중치**를 그대로 로드합니다.
(다시 학습하지 않습니다 — 이것이 "내가 학습시킨 가중치로 추론"입니다.)

In [ ]:
ntn = embeddings.load_ntn()          # ntn.pt (state_dict) 로드 + eval()
print(modeling.describe_model(ntn))

### 1-6. 학습된 NTN으로 이벤트 임베딩 **추론** — 결과 벡터 직접 보기

NVDA 이벤트 3개를 골라 학습된 NTN에 통과시켜 **100차원 이벤트 벡터**를 뽑습니다.

두 가지를 함께 봅니다:
- **raw NTN 출력**: 마지막이 tanh라서 학습이 수렴하면 대부분 ±1 근처로 **포화**됩니다.
- **파이프라인이 실제로 쓰는 벡터**(`event_emb.npy`): s4가 train 구간 기준으로
  차원별 표준화(zero-mean/unit-var)를 해 둔 버전입니다.

> 관찰 포인트: 포화된 tanh 때문에 각 차원은 사실상 **±1 두 값만 갖는 near-binary 코드**처럼 동작합니다.
> 그래서 일부 차원 조각은 서로 다른 이벤트에서도 똑같아 보일 수 있지만, 100차원 전체의
> **pairwise cosine**을 보면 세 이벤트가 뚜렷이 다른 벡터임을 확인할 수 있습니다.

이웃 탐색은 소송 공시 스팸('announce/remind')이 아닌 이벤트로 하고, **동일 triple 중복은 제거**해서
의미상 가까운 서로 다른 이벤트를 확인합니다.

In [ ]:
# 소송 공시 스팸(announce/remind)이 아닌 NVDA 이벤트를 골라야 이웃 탐색이 유의미해집니다.
cand = [int(i) for i in nv_ok if " ".join(ev.p.iloc[int(i)]) not in ("announce", "remind")]
pick = cand[:3]

U = embeddings.embed_events(ntn, ev, pick)     # 사전학습 NTN raw 추론 (tanh 출력)
emb = embeddings.load_event_embeddings()       # 파이프라인용 표준화 임베딩 (44,130 x 100)
print(f"raw NTN 출력 범위: [{U.min():+.3f}, {U.max():+.3f}]  <- tanh 포화(수렴의 흔적)")

# 일부 차원은 이벤트와 무관한 '공유 bias'가 지배합니다(s4가 표준화를 하는 이유).
# 이벤트를 실제로 구분하는, 분산이 큰 차원 8개를 골라 벡터를 비교합니다.
ok_rows = np.where(ok)[0]
var_dims = np.argsort(emb[ok_rows].var(0))[::-1][:8]
print("분산 상위 8개 차원:", var_dims.tolist())
for j, i in enumerate(pick):
    print("\n" + data.triple_str(ev, i))
    print("   raw U[상위분산 8dim]  =", np.round(U[j, var_dims], 3))
    print("   표준화 emb[상위 8dim] =", np.round(emb[i, var_dims], 3))

En3 = emb[pick] / (np.linalg.norm(emb[pick], axis=1, keepdims=True) + 1e-9)
print("\n세 이벤트 간 pairwise cosine (서로 다른 벡터임을 확인):")
print(np.round(En3 @ En3.T, 3))

print("\n첫 이벤트의 cosine 이웃 (중복 triple 제거, 전체 44,130개 중):")
for idx_, cos_, trip in embeddings.nearest_events(ev, emb, pick[0], k=5):
    print(f"   {cos_:+.3f}  {trip}")

rng = np.random.default_rng(13)
sample = rng.choice(np.where(ok)[0], 4000, replace=False)
xy_ev = embeddings.pca_2d(emb[sample])
viz.embedding_scatter(xy_ev, highlight_mask=(ev.ticker.values[sample] == "NVDA"),
                      title="NTN 이벤트 임베딩 PCA (4,000개 샘플, 빨강 = NVDA)")

### 1-7. 예측 모델(EB-CNN) 구조 출력

s6_models.DenseModel은 논문 Sec.3의 head입니다:
- `nn_only=True` → short-term 벡터만 쓰는 **NN** 계열
- `nn_only=False` → long(30일)/mid(7일) 시퀀스에 **narrow conv(l=3) + max-pool**을 적용하는 **CNN** 계열

In [ ]:
from s6_models import DenseModel      # paths.bootstrap() 덕에 flows/에서 import 됨
cnn = DenseModel(100, nn_only=False)
print(modeling.describe_model(cnn))

### 1-8. 학습 코드 잠깐 돌려보기 (진짜 학습, 축약판) + loss 시각화

캐시된 실데이터 특징(`feats_EB.npz`: NTN 이벤트 임베딩의 1/7/30일 텐서)으로
**EB-NN을 6 에폭만** 학습합니다. 에폭별 train BCE loss와 dev MCC를 기록해서 곡선으로 봅니다.

> 공식 재현(s7_train_eval.py)은 4-seed 앙상블 + early-stop으로 돌며,
> 그 결과는 다음 셀의 `results.json`에 이미 저장돼 있습니다.

In [ ]:
hist = modeling.quick_train("EB", nn_only=True, epochs=6)
for i, (l, m) in enumerate(zip(hist["train_loss"], hist["dev_mcc"]), 1):
    print(f"epoch {i}: train BCE {l:.4f} | dev MCC {m:+.4f}")
print(f"\n(단일 시드 축약 학습) test acc {hist['test_acc']:.4f} | test MCC {hist['test_mcc']:+.4f}")
viz.training_curves(hist)

### 1-9. 공식 재현 결과 — 사용자가 학습시킨 모델 매트릭스

`artifacts/results.json` = s7이 저장한 **논문 모델 매트릭스**(Luss, E-NN/CNN, WB-NN/CNN, EB-NN/CNN, TWB/TEB)의
dev/test Accuracy·MCC. majority(0.5355) 점선과 비교해 보세요.

In [ ]:
res = json.load(open(paths.ART / "results.json", encoding="utf-8"))
tbl = pd.DataFrame(res["metrics"]).set_index("model").round(4)
print(tbl.to_string())
print(f"\nn_test={res['n_test']}  majority={res['test_up_rate']:.4f}")
viz.results_bar(res)

### 1-10. 논문식 백테스트 — **사전학습 앙상블의 추론 확률**로

`s53_teb_daily_profit.csv`에는 사용자가 학습시킨 **TEB-CNN 4-seed 앙상블의 test 366일 추론 확률**이 저장돼 있습니다.
이 확률을 실제 s8_simulate 코드(롱 +2% TP / 숏 −1% cover, $10,000/일)에 넣어
- 누적 손익 곡선
- 랜덤 롱/숏 1,000회 대비 p-value
- 확신 임계값 β=0.70 전략

을 계산합니다. **"수익이 나 보인다" ≠ "통계적으로 유의하다"** 를 여기서 처음 만나게 됩니다 → Flow 2로 이어집니다.

In [ ]:
teb = backtest.load_teb_daily()
print(teb.head(3).to_string(index=False))
sim = backtest.paper_simulate(teb.prob_up.values)
print(f"\nalways-trade: 총손익 ${sim['total']:,.0f} / {sim['n_trades']}일")
print(f"(교차검증: CSV 저장 손익 합 = ${teb.profit_always.sum():,.0f})")
viz.equity_curve(sim["dates"], sim["daily"], title="TEB-CNN 앙상블(사전학습 추론) — 논문식 시뮬레이션 누적 손익")
dist, p = backtest.randomization(sim["total"], n=1000)
viz.randomization_hist(dist, sim["total"], p)
sim70 = backtest.paper_simulate(teb.prob_up.values, threshold=0.70)
print(f"beta=0.70: 총손익 ${sim70['total']:,.0f} / 거래 {sim70['n_trades']}회")

---
## Flow 2 — 업그레이드 1: US/NVDA 신호 진단

Flow 1의 결론은 애매합니다: 재현은 됐지만 수익/MCC가 크지 않습니다.
Flow 2는 사용자가 실제로 밟았던 업그레이드 순서(t1·t2 → s10~s27 → s37 → s53)를 따라
**"모델이 약해서인가, 타깃(다음날 방향) 자체가 약해서인가"** 를 실데이터로 판정합니다.

### 2-1. FinBERT 감성 vs 수익률 — 뉴스는 언제 가격에 반영되나

t2_sentiment.py가 계산해 둔 **NVDA 제목 21,576건의 FinBERT 감성 점수**를 일 단위로 평균 내고,
당일 수익률·다음날 수익률과의 상관을 봅니다.

In [ ]:
nv_news = news[news.ticker == "NVDA"].reset_index(drop=True)
sent = np.load(paths.ART / "tf_title_sent.npy")
assert len(sent) == len(nv_news), (len(sent), len(nv_news))
nv_news["sent"] = sent
daily_sent = nv_news.groupby("date")["sent"].mean()
samp = data.load_samples()
joined = samp.set_index("date").join(daily_sent.rename("sent"), how="inner")
same_day = float(joined["sent"].corr(joined["ret"]))
next_day = float(joined["sent"].corr(joined["ret"].shift(-1)))
print(f"corr(당일 감성, 당일 수익률)   = {same_day:+.3f}")
print(f"corr(당일 감성, 다음날 수익률) = {next_day:+.3f}   <- 예측에 쓸 수 있는 쪽")
viz.sentiment_vs_returns(joined["sent"].values[:-1], joined["ret"].shift(-1).dropna().values)

**관찰**: 감성은 **당일** 수익률과는 뚜렷이 상관되지만 **다음날**과는 거의 0입니다.
뉴스는 코인시던트(동시적)이고, 야간 뉴스는 시가 갭에서 이미 소화됩니다 — 이것이 s20~s27 진단의 핵심 발견입니다.

### 2-2. 업그레이드 사다리 — 무엇을 더해도 천장은 0.54~0.56

가격/기술 피처(s10), 선택적 예측(s11), 용량 스윕(s13), 워크포워드(s15)의
**실제 저장 결과 json**에서 test/OOS 정확도를 모아 majority(0.5355)와 비교합니다.

In [ ]:
def _load(name):
    return json.load(open(paths.ART / name, encoding="utf-8"))

boost = _load("results_boost.json")
sel = _load("results_selective.json")
wf = _load("results_walkforward.json")
cap = _load("results_capacity.json")

ladder = {
    "s7 논문 매트릭스 best": max(r["test_acc"] for r in res["metrics"]),
    "s10 가격+뉴스 GBM best": float(boost["best_full_test_acc"]),
    "s11 selective (cov 0.5)": float(next(r["test_acc"] for r in sel["dev_quantile_rule"]
                                          if r["model"] == "price+news" and r["target_cov"] == 0.5)),
    "s13 capacity best": max(r["test"] for r in cap),
    "s15 walk-forward OOS": float(wf["oos_full_acc"]),
}
for k_, v_ in ladder.items():
    print(f"{k_:26s} {v_:.4f}")
viz.upgrade_ladder(ladder, baseline=res["test_up_rate"])

### 2-3. 생존성 감사(s53) — "+$1,260"은 살아남는가

report.md의 TEB-CNN +$1,260은 매력적으로 보였지만, s53이 4-seed·bootstrap·다중검정으로 재검증했습니다.
아래에서 **시드별 손익 분산**, **앙상블 p-value**, **Bonferroni 보정**, 그리고
**dev로 고른 임계값 vs test 임계값 곡선**을 확인하세요.

In [ ]:
sv = kr.load_survival()
print("시드별 always-trade 손익:")
for r in sv["seed_rows"]:
    print(f"  seed {r['seed']}: dev MCC {r['dev_mcc']:+.3f} | test profit ${r['test_profit_always']:,.0f}")
ens, al, rd, mt = sv["ensemble_metrics"], sv["always"], sv["randomization"], sv["multiple_testing"]
print(f"\n4-seed 앙상블: test acc {ens['test_acc']:.4f} | MCC {ens['test_mcc']:+.4f}")
print(f"always-trade 손익 ${al['profit_total']:,.0f} | bootstrap CI95 "
      f"[{al['bootstrap_profit_ci95'][0]:,.0f}, {al['bootstrap_profit_ci95'][2]:,.0f}] | "
      f"P(<=0)={al['bootstrap_prob_profit_le_0']:.3f}")
print(f"randomization p = {rd['p_ge_always']:.3f} | Bonferroni(11 models) p = {mt['teb_bonferroni_11_models']:.2f}")
dev_c = pd.read_csv(paths.ART / "s53_teb_dev_threshold_curve.csv")
test_c = pd.read_csv(paths.ART / "s53_teb_test_threshold_curve.csv")
viz.threshold_curves(dev_c, test_c)

**Flow 2 결론**: 표현(FinBERT)·피처·검증을 아무리 강화해도 **다음날 방향 타깃**은
majority ±0.02 안에서 끝났고, 좋아 보였던 수익은 시드/정렬/임계값 선택 노이즈였습니다.
→ 모델이 아니라 **타깃 정의를 바꿔야 한다**는 것이 Flow 3의 출발점입니다.

---
## Flow 3 — 업그레이드 2: KR + HIGH/LOW 재정의 + 실거래성 검증

### 3-1. KR 데이터 스케일 + 타깃 재정의의 근거

한국 시장 626종목 × 2015~2026 일봉(164만 행)에서
**시가 대비 고가/저가 초과(exceedance) 발생률**을 직접 계산합니다.
"다음날 방향"과 달리 이 타깃은 **시가 시점에 베팅이 정의**되고, 뉴스가 실제로 담고 있는
**변동성/주목도 정보**와 정합적입니다.

In [ ]:
ko = kr.load_kr_ohlcv()
print(f"KR OHLCV: {len(ko):,}행 | {ko.ticker.nunique()}종목 | "
      f"{ko.date.min().date()} ~ {ko.date.max().date()}")
rates = kr.exceedance_rates(ko)
for k_, v_ in rates.items():
    print(f"  {k_}: {v_:.3%}")
viz.exceedance_bars(rates)

### 3-2. 사용자 학습 GBM의 추론 점수 — 랭킹 lift는 진짜인가

`kr52_scores_full.parquet`는 **2021-11 이전 데이터로만 학습한 GBM**이
그 이후 753,893 종목-일에 대해 낸 **모델-OOS UP5 점수**입니다.
날짜별 top-1/top-3만 골랐을 때 실제로 +5% 고가 터치가 얼마나 자주 나왔는지(base 대비)를 계산합니다.

In [ ]:
sc = kr.load_kr_scores()
print(f"scores: {len(sc):,}행 | {sc.date.min().date()} ~ {sc.date.max().date()}")
top1 = kr.daily_topk_hit(sc, k=1)
top3 = kr.daily_topk_hit(sc, k=3)
for name, t in (("top-1", top1), ("top-3", top3)):
    print(f"{name}: hit {t['hit_rate']:.3f} vs base {t['base_rate']:.3f} "
          f"({t['hit_rate']/t['base_rate']:.1f}x lift, {t['n_days']:,}일)")
viz.score_lift({"top-1/day": top1, "top-3/day": top3})

### 3-3. 그런데 돈이 되는가 — 실거래 백테스트 3종 (사용자 산출물)

랭킹이 진짜라도 수익은 별개입니다. 사용자가 이미 돌려 둔 세 백테스트의 equity를 그대로 로드합니다.
- **s50**: 5분봉으로 TP/SL 선후관계를 확정한 OOS 백테스트 → 음(-)
- **s51**: 딥/돌파/지연 등 6개 현실적 엔트리 최적화 → 전부 음(-)
- **s52**: 스탑 없는 EXACT 테스트(경로 가정 0) → EV ≈ 0, 시가가 이미 다 반영

In [ ]:
cur = kr.load_equity_curves()
for name, df_ in cur.items():
    print(f"{name}: 시작 {float(df_.equity.iloc[0]):.3f} -> 최종 {float(df_.equity.iloc[-1]):.3f}")
viz.kr_equity_panels(cur)

### 3-4. 갭 타깃(s54)과 XAI(s55) — 모델은 어떤 뉴스를 보고 판단했나

- s54: "1일 뉴스 버킷" 정의(장중/야간/분리)에 따라 **시가 갭 방향** 예측이 어떻게 변하는지
- s55: KR-FinBERT 임베딩 모델이 **가장 크게 맞춘 날 / 틀린 날**에 기여한 실제 뉴스 제목

In [ ]:
g = kr.load_gap_results()
print(f"갭 실험: 링크 {g['n_links']:,} | 샘플 {g['n_samples']:,} | test gap-up base {g['test_gap_up_rate']:.3f}")
rows = []
for name, v in g["variants"].items():
    d = v.get("direction", {})
    rows.append({"variant": name, "tradable": v.get("tradable"),
                 "acc": d.get("acc"), "base": d.get("base"), "mcc": d.get("mcc")})
print(pd.DataFrame(rows).round(4).to_string(index=False))

x = kr.load_xai()
print(f"\nXAI encoder: {x['encoder']}")
print("\n[가장 크게 맞춘 날 — 근거 뉴스]")
for d in x["xai_best_days"][:3]:
    top = d["top_titles"][0]["title"] if d.get("top_titles") else "(없음)"
    print(f"  {d['ticker']} {d['date']} gap {d['gap_pct']:+.1f}% :: {top}")
print("[가장 크게 틀린 날 — 근거 뉴스]")
for d in x["xai_worst_days"][:3]:
    top = d["top_titles"][0]["title"] if d.get("top_titles") else "(없음)"
    print(f"  {d['ticker']} {d['date']} gap {d['gap_pct']:+.1f}% :: {top}")

---
## 마무리 — 세 Flow가 말하는 것

In [ ]:
summary = {
    "Flow 1 논문 재현": {
        "best test acc (s7)": round(max(r["test_acc"] for r in res["metrics"]), 4),
        "majority baseline": round(res["test_up_rate"], 4),
        "TEB 앙상블 백테스트": f"${sim['total']:,.0f} (randomization p={p:.3f})",
    },
    "Flow 2 US 진단": {
        "업그레이드 사다리 최고": round(max(ladder.values()), 4),
        "walk-forward OOS": round(float(wf["oos_full_acc"]), 4),
        "TEB Bonferroni p": mt["teb_bonferroni_11_models"],
    },
    "Flow 3 KR HIGH/LOW": {
        "daily top-1 hit vs base": f"{top1['hit_rate']:.3f} vs {top1['base_rate']:.3f}",
        "실거래 결론": "랭킹은 진짜지만 s50/s51 음(-), s52 EV~0 — 시가가 이미 반영",
    },
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

**핵심 교훈**
1. **재현 성공 ≠ 신호 존재** — Flow 1은 논문 구조를 그대로 재현하지만, 다음날 방향의 정보량은 얇습니다.
2. **검증이 성능보다 먼저** — Flow 2의 워크포워드/생존성 감사가 "+$1,260"을 노이즈로 판정했습니다.
3. **타깃 정의가 모델보다 중요** — Flow 3에서 HIGH/LOW로 바꾸자 랭킹 lift(7~50%+)는 진짜가 됐지만,
   실거래 EV는 시가에 이미 반영돼 0 근처였습니다. 예측력과 수익화는 다른 문제입니다.

**후속 과제 제안**
1. `modeling.quick_train`을 `rep="TEB"`, `nn_only=False`로 바꿔 CNN 계열을 짧게 학습해 보고 dev MCC 곡선을 비교하세요.
2. `backtest.paper_simulate`의 `threshold`를 0.55~0.90으로 스윕해 **dev에서 고른 값**이 test에서도 최선인지 확인하세요.
3. `kr.daily_topk_hit`의 `tp`를 0.02/0.03으로 바꿔 UP2/UP3 랭킹 lift를 재계산하고, s52의 EV~0 결론이 유지되는지 논증해 보세요.